# Stock Market Volatility Classification Using News Sentiment
**MSc in Data Science & Analytics Thesis (EE648) — Maynooth University**  
**Author:** Uthej Alagani | **Supervisor:** Dr. Bryan Hennelly | **Grade:** First Class Honours (1:1)

---

### End-to-End Pipeline Overview
1. **FNSPID NASDAQ-100 Data Filtering & Cleaning** (`55,349` aligned news + daily OHLCV records across `AAPL`, `MSFT`, `AMZN`, `CRM`, `NVDA`, `NOW`, `TSLA`, `ADBE`, `INTC`, `AVGO`).
2. **Domain Feature Engineering & Per-Symbol Normalization**:
   - Return Momentum: $R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$ and 15-day lags (`Return_Lag_1..15`)
   - 3-Day Rolling Volatility: $\sigma_t^{(3)} = \text{std}(R_{t-2}, R_{t-1}, R_t)$
   - 3-Day Average True Range (ATR): $\text{ATR}_t^{(3)} = \frac{1}{3} \sum_{i=t-2}^{t} (H_i - L_i) \div C_i$
   - Calendar Signals (`Day_of_Week`, `Month`, `Year`) & Per-Symbol `StandardScaler`
3. **FinBERT Financial Sentiment Extraction** (`ProsusAI/finbert`, `safe_truncate(max_tokens=512)`, `Sentiment_Lag_1..15`).
4. **PyTorch 1D-CNN with Multihead Attention (`CNNAttention`)**: **82.11% Accuracy**, **0.82 ROC-AUC**.
5. **SMOTE + XGBoost Classifier (`XGBClassifier`)**: **84.10% Accuracy**, **0.86 ROC-AUC**.

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Figure 4.2: FinBERT Sentiment Extraction Pipeline
# from transformers import pipeline, AutoTokenizer
# model_name = 'ProsusAI/finbert'
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# sentiment_model = pipeline('sentiment-analysis', model=model_name, tokenizer=tokenizer)

def safe_truncate(text, tokenizer, max_tokens=512):
    encoded = tokenizer.encode(text, truncation=True, max_length=max_tokens, return_tensors=None)
    return tokenizer.decode(encoded, skip_special_tokens=True)

In [2]:
# Chapter 4.4: PyTorch 1D-CNN with Multihead Attention Architecture
import torch
import torch.nn as nn

class CNNAttention(nn.Module):
    def __init__(self, input_size, n_classes=2):
        super(CNNAttention, self).__init__()
        self.conv = nn.Conv1d(1, 64, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.attn = nn.MultiheadAttention(embed_dim=64, num_heads=4, batch_first=True)
        self.fc1 = nn.Linear(64 * input_size, 128)
        self.out = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (batch_size, 1, sequence_length)
        x = self.relu(self.conv(x))
        x = x.permute(0, 2, 1)  # (batch_size, sequence_length, channels)
        attn_out, _ = self.attn(x, x, x)
        attn_out = attn_out.reshape(attn_out.size(0), -1)
        x = self.relu(self.fc1(attn_out))
        return self.out(x)

In [3]:
# Chapter 4.5: SMOTE + XGBoost Classifier for Binary Volatility Prediction
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

# Apply SMOTE exclusively on the training partition
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

# xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
# xgb_model.fit(X_train_resampled, y_train_resampled)